# Problem 3: Persona Chatbot

**Objective:** build a chatbot web application with an original persona, powered by OpenAI's open-weight gpt-oss model.

At the end of the session, the team will have a chat application with a **public link** that can be opened on any laptop or mobile phone.

**Learning outcomes:** understanding what a *system prompt* is, what *temperature* controls, and how a chat application sends a conversation to an AI model.

### How to use this notebook
- A notebook is made up of **cells**. To run a cell, select it and press **Shift + Enter** (or select the run button).
- On the first run, run the cells **in order, from top to bottom**.
- Comments marked **STUDENT TODO** show where changes are expected:
  - **Level 1:** text changes. No coding experience required.
  - **Level 2:** changes to settings and values.
  - **Level 3:** new features. Mentors are available to help.
- After changing a cell, **re-run that cell and then re-run the "Launch the application" cell** for the change to take effect.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

### Obtaining a free Groq API key (one per team; about 2 minutes)

Groq is a free service that hosts **open-weight AI models**, such as OpenAI's gpt-oss. The code in this notebook sends a message to Groq and receives the model's response.

1. Go to **https://console.groq.com** and sign in with a Google or GitHub account.
2. Select **API Keys** → **Create API Key**, enter any name, and **copy** the key (it begins with `gsk_`).
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GROQ_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it. The Secrets panel keeps it private.
>
> **No key yet:** the notebook runs in **demonstration mode** with sample responses, so you can explore and edit it before a key is available.

## Step 0: Installation and setup (run both cells)

In [ ]:
# Installs the required libraries. This takes about 30 seconds; the output text is expected.
!pip install -q "gradio>=5,<6" openai

In [ ]:
# SETUP: run this cell once. No changes are required here.
import os
import openai
from openai import OpenAI

def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

GROQ_API_KEY = _get_secret("GROQ_API_KEY")
DEMO_MODE = not GROQ_API_KEY

# Groq uses the same interface as the OpenAI API, so the openai library is used
# and pointed at Groq's servers.
client = None if DEMO_MODE else OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

def _friendly_error(e):
    if isinstance(e, openai.AuthenticationError):
        return "Error: the GROQ_API_KEY was rejected. Check that the complete key (beginning with gsk_) is saved in Colab Secrets, then re-run the setup cell."
    if isinstance(e, openai.RateLimitError):
        return "Error: the Groq free-tier usage limit has been reached. Wait about one minute and try again, or set MODEL = 'openai/gpt-oss-20b'."
    if isinstance(e, openai.NotFoundError):
        return f"Error: Groq does not recognise the model '{MODEL}'. Check the spelling; the model may also have been retired. See https://console.groq.com/docs/models"
    if isinstance(e, openai.APIConnectionError):
        return "Error: unable to connect to Groq. Check the internet connection and try again."
    return f"Error: the request to the AI service failed ({e})."

def ask_llm(messages, temperature=0.7, json_mode=False, demo_reply="(Demonstration mode response)"):
    """Sends a list of chat messages to the AI model and returns its response as text.
    In demonstration mode, returns `demo_reply` instead. If the request fails,
    returns a message beginning with "Error:" instead of stopping the program."""
    if DEMO_MODE:
        return demo_reply
    try:
        extra = {"response_format": {"type": "json_object"}} if json_mode else {}
        if MODEL.startswith("openai/gpt-oss"):
            extra["reasoning_effort"] = "low"  # gpt-oss models reason before answering; "low" keeps responses fast
        response = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature, **extra
        )
        return response.choices[0].message.content
    except Exception as e:
        return _friendly_error(e)

if DEMO_MODE:
    print("Demonstration mode: no GROQ_API_KEY was found, so the application will show sample responses.")
    print("To connect the AI model: Secrets (left sidebar) → add GROQ_API_KEY → enable Notebook access → re-run this cell.")
else:
    print("Connected to Groq. The AI model is ready.")

## Step 1: Define the persona

This is the most important cell. The **system prompt** (`PERSONA`) contains instructions that the AI reads before every conversation. It sets **who the chatbot is, how it communicates, and what it will and will not do.**

Example concepts: a campus orientation guide for first-year students, a history tutor with a distinctive voice, a study-planning assistant, a subject-specific revision helper, or a career guidance assistant.

**Guidance:** an effective persona states *who* the chatbot is, *how* it communicates, *what* it helps with, and *what* it should decline. Specific instructions produce better results than general ones.

In [ ]:
# STUDENT TODO (Level 1): set the chatbot's name and persona.
BOT_NAME = "Professor Byte"

PERSONA = """
You are Professor Byte, a patient computer science tutor for first-year college students.
- Explain every idea using a simple comparison from everyday life.
- Keep answers short: no more than 4 sentences.
- Use plain language and avoid unexplained technical terms.
- If a question is not about computers or programming, politely explain that you can only help with computing topics.
"""

# STUDENT TODO (Level 1): the first message users see when they open the application.
GREETING = "Hello, I am Professor Byte. Ask me any question about computers or programming."

# STUDENT TODO (Level 1): example questions, shown as selectable options below the chat.
EXAMPLES = [
    "What is a variable?",
    "Why does code have bugs?",
    "How does the internet work?",
]

# STUDENT TODO (Level 2): temperature controls how varied the responses are.
#   0.0  consistent and predictable; the same answer each time
#   0.7  balanced (default)
#   1.5  highly varied, and sometimes inaccurate
TEMPERATURE = 0.7

# STUDENT TODO (Level 2): the number of previous messages the chatbot remembers.
#   Higher values give better memory but are slower and use more of the free limit.
MEMORY = 10

# STUDENT TODO (Level 2): select the open-weight model that generates responses.
#   "openai/gpt-oss-20b"   OpenAI gpt-oss, 20 billion parameters. Fast and suitable for most tasks. (default)
#   "openai/gpt-oss-120b"  OpenAI gpt-oss, 120 billion parameters. More capable, slower, lower limits.
#   "qwen/qwen3.8-27b"     Alibaba Qwen (a preview model, which Groq may withdraw at short notice).
#   Groq retires models from time to time. Current list: https://console.groq.com/docs/models
MODEL = "openai/gpt-oss-20b"

print(f"{BOT_NAME} is configured.")

## Step 2: The chat function (run this cell)

This function runs every time a user sends a message. It builds a list of messages (the system prompt first, then the recent conversation history, then the new message) and sends it to the AI model. No changes are needed yet, but the comments explain how it works.

In [ ]:
def chat(message, history):
    # 1. Start with the system prompt (the persona).
    messages = [{"role": "system", "content": PERSONA}]

    # 2. Add the most recent messages so the chatbot keeps the context of the conversation.
    for turn in history[-MEMORY:]:
        messages.append({"role": turn["role"], "content": str(turn["content"])})

    # 3. Add the user's new message.
    messages.append({"role": "user", "content": message})

    # STUDENT TODO (Level 3): modify the message before it is sent to the AI.
    #   Example: messages[-1]["content"] = message + " (Answer in exactly 3 bullet points.)"

    # 4. Send the messages to the AI and return its response.
    reply = ask_llm(
        messages,
        temperature=TEMPERATURE,
        demo_reply=f"(Demonstration mode) This is {BOT_NAME}. Add a GROQ_API_KEY to enable AI responses. Your message was: \"{message}\"",
    )

    # STUDENT TODO (Level 3): modify the response before it is shown to the user.
    #   Example: reply = reply + "\n\nWould you like an example?"
    return reply

# Quick test: send one message to the chatbot before building the application.
print(f"User: {EXAMPLES[0]}")
print(f"{BOT_NAME}: {chat(EXAMPLES[0], [])}")

## Step 3: Launch the application

Run this cell. The application appears **below the cell**, and a **public link** (`https://....gradio.live`) is also created, which can be shared and opened on a mobile phone.

After changing any cell above, **re-run that cell and then re-run this one.**

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title=f"Chat with {BOT_NAME}",
    # STUDENT TODO (Level 1): describe the chatbot in one sentence.
    description="A computer science tutor that explains concepts using everyday comparisons. Built at the Hacktoberfest Hack Day with the open-weight gpt-oss model.",
    chatbot=gr.Chatbot(value=[{"role": "assistant", "content": GREETING}], type="messages", height=450),
    examples=EXAMPLES,
    # STUDENT TODO (Level 2): alternative themes: gr.themes.Soft(), gr.themes.Glass(), gr.themes.Monochrome(), gr.themes.Ocean()
    theme=gr.themes.Soft(),
)

demo.launch(share=True)

## Level 3: Optional extensions

These are larger changes. Attempt them one at a time and ask a mentor for help where needed.

**1. Temperature comparison (recommended for the demonstration).** Ask the same question three times at `TEMPERATURE = 0.0`, then three times at `1.5`. Record how the responses differ.

**2. Larger model.** Change `MODEL` to `"openai/gpt-oss-120b"`. Assess whether it follows the persona more consistently, and whether it is slower.

**3. Persona selection.** Allow users to choose between several personas from a menu.
<details><summary>Hint (select to expand)</summary>

```python
PERSONAS = {
    "Professor Byte": "You are a patient computer science tutor...",
    "Campus Guide": "You are a helpful guide for first-year students...",
}

def chat(message, history, persona_name):        # new third parameter
    messages = [{"role": "system", "content": PERSONAS[persona_name]}]
    ...  # the rest of the function is unchanged

# In gr.ChatInterface(...), add:
#   additional_inputs=[gr.Dropdown(list(PERSONAS), value=list(PERSONAS)[0], label="Select a persona")],
# and change examples to a list of lists: [["What is a variable?", list(PERSONAS)[0]], ...]
```
</details>

**4. Robustness testing.** Attempt to make the chatbot abandon its persona or discuss unrelated topics. Then revise `PERSONA` so that it resists these attempts. Record the attempts that succeeded and the corrections made, for use in the demonstration.

**5. Restricted word.** Add an instruction that the chatbot must never use a particular word, and invite the audience to try to make it do so.

**6. Response review.** After the chatbot responds, call `ask_llm` a second time with a different system prompt (for example: "You are a strict examiner. Rate this answer from 1 to 10 for correctness and explain your rating.") and add the rating to the response.

## Demonstration checklist
- [ ] The public link works on a mobile phone
- [ ] The chatbot stays consistent with its persona for at least five exchanges
- [ ] The team can explain **system prompt** and **temperature** in one sentence each
- [ ] One example of robustness testing and the correction made

## Troubleshooting

| Issue | Resolution |
|---|---|
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The application does not reflect recent changes | Re-run the edited cell **and** the "Launch the application" cell. |
| The notebook remains in demonstration mode | Add `GROQ_API_KEY` in Secrets, enable **Notebook access**, and re-run the setup cell. |
| "Usage limit reached" error | Wait about one minute. Free keys allow a limited number of requests per minute. |
| `SyntaxError` | Usually caused by a missing quotation mark `"`, bracket `)` or comma. Check the line shown in the error. |
| Unexpected behaviour | Select **Runtime → Restart session**, then run all cells from the top. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |